# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides an end-to-end guide for loading, exploring, and analyzing the FAIR<sup>2</sup> dataset using the `mlcroissant` library. All record sets, fields, and columns are referenced by their `@id` as required by the Croissant standard.

### Dataset Source
The dataset metadata (Croissant schema) is provided at the following URL ([view schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)):

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display dataset metadata (name, description)
meta = dataset.metadata
print(f"Dataset title: {meta.name}")
print(f"Description: {meta.description}")
print(f"Version: {meta.version}")
print(f"Published: {getattr(meta,'datePublished',None)}")
print(f"License: {meta.license}")

## 2. Data Overview
Let's enumerate all available record sets and their field (column) `@id`s in the dataset.

We use `record_set` `@id` and `field` (column) `@id` for all referencing, as per the Croissant specification.

In [ ]:
# List available record sets and their columns (fields)

recordsets = dataset.metadata.record_sets
print(f"Total record sets: {len(recordsets)}\n")
for rset in recordsets:
    print(f"=== RecordSet @id: {rset['@id']} ===")
    # List fields (columns) with @id and name
    for field in rset.get('field', []):
        # Some field may be a dict or str, try to access @id
        if isinstance(field, dict):
            field_id = field.get('@id','')
            field_name = field.get('name','')
        else:
            # Sometimes field is just @id, try to fetch Field object by @id
            field_id = field
            # Find the actual Field object in the dataset metadata
            field_obj = next((f for f in dataset.metadata.fields if f['@id']==field_id), {})
            field_name = field_obj.get('name','')
        print(f"  field @id: {field_id}, name: {field_name}")
    print('---')

## 3. Data Extraction
Load available data for each record set into separate DataFrames, using their `@id`.

> **Note:** Please consult the printed overview above for correct `@id` values. Some record sets may not have data or may not be accessible directly if, for example, they reference data files requiring additional access.


In [ ]:
# Example: Collect list of record set @ids from metadata
record_sets = [rset['@id'] for rset in dataset.metadata.record_sets]
dataframes = {}

# Load records for each record set
for record_set_id in record_sets:
    try:
        print(f"Loading records for RecordSet @id: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            print(f"  Loaded {len(df)} rows, columns: {df.columns.tolist()}")
        else:
            print("  No data found.")
            df = pd.DataFrame()
        dataframes[record_set_id] = df
    except Exception as e:
        print(f"  ERROR loading: {e}")
        dataframes[record_set_id] = pd.DataFrame()
    print()
# Choose the main record set (the largest one with data) for further analysis
main_record_set_id = max(dataframes, key=lambda k: len(dataframes[k]))
print(f"Main record set for analysis: {main_record_set_id}")
print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's perform some basic analysis using numeric and categorical fields. We'll use field `@id` and group by or filter accordingly.

If you want to explore a different numeric or categorical field, look at the columns shown above.

In [ ]:
# Select a numeric field by its @id (replace with the actual @id you want to use)
# E.g. numeric_field_id = 'cr:coverage_percentage'

numeric_field_id = None
for col in dataframes[main_record_set_id].columns:
    if 'coverage' in col.lower() or 'count' in col.lower() or 'mw' in col.lower():
        numeric_field_id = col
        break

if numeric_field_id is None:
    print("No obvious numeric field detected, using first column.")
    numeric_field_id = dataframes[main_record_set_id].columns[0]

# Filter for numeric values only
df = dataframes[main_record_set_id]
df_numeric = df.copy()
df_numeric[numeric_field_id] = pd.to_numeric(df_numeric[numeric_field_id], errors="coerce")
threshold = df_numeric[numeric_field_id].mean() if df_numeric[numeric_field_id].notnull().sum()>0 else 0
filtered_df = df_numeric[df_numeric[numeric_field_id] > threshold]
print(f"Filtered to records where {numeric_field_id} > {threshold:.2f} [{filtered_df.shape[0]} / {df_numeric.shape[0]} rows].")

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to group by a likely categorical/group field
group_field = None
for col in df.columns:
    if col != numeric_field_id and df[col].nunique()<=10:
        group_field = col
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
    print(f"\nGrouped average {numeric_field_id} by {group_field}:")
    print(grouped_df.head())
else:
    print("No suitable group field found for aggregation.")

## 5. Visualization
Let's visualize the distribution of our selected numeric field and, if available, compare averages by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.histplot(filtered_df[numeric_field_id].dropna(), bins=20, kde=True)
plt.title(f"Distribution of numeric field: {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

if group_field:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field, y=numeric_field_id, data=filtered_df)
    plt.title(f"{numeric_field_id} distribution by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

In this notebook, we loaded the FAIR^2 Mass Spectrometry dataset using the `mlcroissant` library, inspected its record sets and fields by `@id`, extracted available data into DataFrames, and performed initial exploratory analysis.

- We demonstrated referencing all dataset components by Croissant `@id`.
- Basic data preparation (filtering, normalization) and summary statistics were shown.
- Data visualization (histogram, and grouped boxplot if applicable) served to illustrate data quality and trends.

For deeper analysis, consult the record set and field documentation in the Croissant schema or [browse the metadata directly](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

You can now build on this notebook for more advanced exploration, modeling, or to integrate with your research workflows.